In [1]:
# ============================================================
# BUILD thesis_final_master.csv
# STRICT APPEND OF RECENT HIGHER-QUALITY RELEASED MOVIES
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import ast
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
PROJECT_DIR = "/content/drive/MyDrive/thesis_movie_project"
RAW_DIR = f"{PROJECT_DIR}/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/processed"

OLD_MASTER_PATH = f"{PROCESSED_DIR}/thesis_master.csv"
NEW_DATASET_PATH = f"{RAW_DIR}/IMDB TMDB Movie Metadata Big Dataset (1M).csv"
OUTPUT_PATH = f"{PROCESSED_DIR}/thesis_master_final.csv"

# Strict append settings
MIN_APPEND_YEAR = 2017
MIN_RUNTIME = 60            # remove short/noisy entries
MIN_VOTE_COUNT = 10         # quality threshold
MIN_POPULARITY = 1.0        # light visibility threshold

FINAL_COLS = [
    "id",
    "movieId",
    "imdb_id",
    "title",
    "overview",
    "keywords_clean",
    "cast_clean",
    "director_name",
    "genres_clean",
    "original_language",
    "release_date",
    "release_year",
    "budget",
    "revenue",
    "roi_raw",
    "runtime",
    "popularity",
    "vote_average",
    "vote_count",
    "text_overview_only",
    "text_semantic_enriched",
    "text_semantic_full",
]

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def require_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

def clean_text(s):
    return (
        s.fillna("")
         .astype(str)
         .str.replace(r"\s+", " ", regex=True)
         .str.strip()
    )

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def parse_listlike_cell(x):
    """
    Parse strings like:
    "['Action', 'Drama']"
    "[]"
    NaN
    into a Python list of strings.
    """
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    x = str(x).strip()
    if x == "" or x == "[]" or x.lower() == "nan":
        return []
    try:
        val = ast.literal_eval(x)
        if isinstance(val, list):
            out = []
            for item in val:
                if isinstance(item, dict):
                    name = str(item.get("name", "")).strip()
                    if name:
                        out.append(name)
                else:
                    item = str(item).strip()
                    if item:
                        out.append(item)
            return out
        return []
    except Exception:
        return []

def list_to_clean_string(x, top_n=None):
    vals = parse_listlike_cell(x)
    if top_n is not None:
        vals = vals[:top_n]
    return " ".join(vals).strip()

# ------------------------------------------------------------
# 1. LOAD OLD MASTER
# ------------------------------------------------------------
print("1. Loading old master...")
require_file(OLD_MASTER_PATH)
old_master = pd.read_csv(OLD_MASTER_PATH, low_memory=False)

for col in FINAL_COLS:
    if col not in old_master.columns:
        if col == "movieId":
            old_master[col] = pd.Series([pd.NA] * len(old_master), dtype="Int64")
        else:
            old_master[col] = np.nan

old_master = old_master[FINAL_COLS].copy()
old_master["id"] = to_num(old_master["id"])
old_master = old_master.dropna(subset=["id"]).copy()
old_master["id"] = old_master["id"].astype("int64")
old_master["imdb_id"] = clean_text(old_master["imdb_id"])
old_master["title"] = clean_text(old_master["title"])
old_master["release_year"] = to_num(old_master["release_year"])

print("Old master shape:", old_master.shape)

# Match keys
old_ids = set(old_master["id"].dropna().astype(int))
old_imdb = set(old_master["imdb_id"][old_master["imdb_id"].str.len() > 0].tolist())

old_master["title_key"] = old_master["title"].str.lower().str.strip()
old_title_year = set(
    zip(
        old_master["title_key"],
        old_master["release_year"].fillna(-1).astype(int)
    )
)

# ------------------------------------------------------------
# 2. LOAD NEW DATASET
# ------------------------------------------------------------
print("\n2. Loading new dataset...")
require_file(NEW_DATASET_PATH)
new_df = pd.read_csv(NEW_DATASET_PATH, low_memory=False)
print("New dataset shape:", new_df.shape)

# ------------------------------------------------------------
# 3. BASE FILTERS: RELEASED + RECENT + NON-FUTURE
# ------------------------------------------------------------
print("\n3. Applying base release/year filters...")

new_df["release_date"] = pd.to_datetime(new_df["release_date"], errors="coerce")
if "release_year" in new_df.columns:
    new_df["release_year"] = to_num(new_df["release_year"])
else:
    new_df["release_year"] = new_df["release_date"].dt.year

today = pd.Timestamp.today().normalize()

base = new_df.copy()

if "status" in base.columns:
    base = base[base["status"].astype(str).str.strip().str.lower() == "released"]

base = base[base["release_date"].notna()]
base = base[base["release_date"] <= today]
base = base[base["release_year"] >= MIN_APPEND_YEAR]

print("After release/year filter:", base.shape)

# ------------------------------------------------------------
# 4. STANDARDIZE NEW DATASET TO YOUR SCHEMA
# ------------------------------------------------------------
print("\n4. Standardizing fields...")

# keys / identifiers
base["id"] = to_num(base["id"])
base = base.dropna(subset=["id"]).copy()
base["id"] = base["id"].astype("int64")

base["imdb_id"] = clean_text(base["imdb_id"])
base["title"] = clean_text(base["title"])
base["overview"] = clean_text(base["overview"])
base["original_language"] = clean_text(base["original_language"])

# fallback text sources
if "keywords" in base.columns:
    kw_primary = base["keywords"].apply(lambda x: list_to_clean_string(x))
else:
    kw_primary = pd.Series([""] * len(base), index=base.index)

if "all_combined_keywords" in base.columns:
    kw_fallback = base["all_combined_keywords"].apply(lambda x: list_to_clean_string(x))
else:
    kw_fallback = pd.Series([""] * len(base), index=base.index)

base["keywords_clean"] = np.where(
    clean_text(kw_primary).str.len() > 0,
    clean_text(kw_primary),
    clean_text(kw_fallback)
)

if "Cast_list" in base.columns:
    base["cast_clean"] = base["Cast_list"].apply(lambda x: list_to_clean_string(x, top_n=5))
else:
    base["cast_clean"] = ""

if "Director" in base.columns:
    base["director_name"] = clean_text(base["Director"])
else:
    base["director_name"] = ""

if "genres_list" in base.columns:
    base["genres_clean"] = base["genres_list"].apply(lambda x: list_to_clean_string(x))
else:
    base["genres_clean"] = ""

for col in ["budget", "revenue", "runtime", "popularity", "vote_average", "vote_count"]:
    if col in base.columns:
        base[col] = to_num(base[col])
    else:
        base[col] = np.nan

# finance cleaning
base["budget"] = base["budget"].where(base["budget"] > 0, np.nan)
base["revenue"] = base["revenue"].where(base["revenue"] > 0, np.nan)

# other numeric sanity
base["runtime"] = base["runtime"].where(base["runtime"] > 0, np.nan)
base["popularity"] = base["popularity"].where(base["popularity"] >= 0, np.nan)
base["vote_average"] = base["vote_average"].where(
    (base["vote_average"] >= 0) & (base["vote_average"] <= 10), np.nan
)
base["vote_count"] = base["vote_count"].where(base["vote_count"] >= 0, np.nan)

# roi
base["roi_raw"] = np.where(
    base["budget"].notna() & base["revenue"].notna(),
    base["revenue"] / base["budget"],
    np.nan
)

# old master had movieId bridge, new data doesn't
base["movieId"] = pd.Series([pd.NA] * len(base), dtype="Int64")

# ------------------------------------------------------------
# 5. STRICT QUALITY FILTERS FOR APPENDED MOVIES
# ------------------------------------------------------------
print("\n5. Applying strict quality filters...")

# Require useful semantic text
base = base[
    (clean_text(base["title"]).str.len() > 0) &
    (clean_text(base["overview"]).str.len() > 30)
].copy()

# Require reasonable movie-like runtime
base = base[base["runtime"].notna() & (base["runtime"] >= MIN_RUNTIME)].copy()

# Require at least some signal of significance / visibility
quality_mask = (
    (base["vote_count"].fillna(0) >= MIN_VOTE_COUNT) |
    (base["popularity"].fillna(0) >= MIN_POPULARITY) |
    (base["revenue"].notna()) |
    (base["budget"].notna())
)

base = base[quality_mask].copy()

# dedupe inside new data
base = base.drop_duplicates(subset="id").copy()

print("After strict filters:", base.shape)

# ------------------------------------------------------------
# 6. BUILD TEXT COLUMNS
# ------------------------------------------------------------
print("\n6. Building text columns...")

base["text_overview_only"] = clean_text(base["overview"])

base["text_semantic_enriched"] = (
    clean_text(base["title"]) + " " +
    clean_text(base["keywords_clean"]) + " " +
    clean_text(base["overview"])
).str.replace(r"\s+", " ", regex=True).str.strip()

base["text_semantic_full"] = (
    clean_text(base["title"]) + " " +
    clean_text(base["keywords_clean"]) + " " +
    clean_text(base["cast_clean"]) + " " +
    clean_text(base["overview"])
).str.replace(r"\s+", " ", regex=True).str.strip()

base = base[FINAL_COLS].copy()

# ------------------------------------------------------------
# 7. REMOVE OVERLAPS WITH OLD MASTER
# ------------------------------------------------------------
print("\n7. Removing overlaps with old master...")

base["imdb_id"] = clean_text(base["imdb_id"])
base["title_key"] = clean_text(base["title"]).str.lower().str.strip()
base["release_year"] = to_num(base["release_year"])

mask_not_old_id = ~base["id"].isin(old_ids)
mask_not_old_imdb = ~base["imdb_id"].isin(old_imdb)

new_title_year = list(zip(
    base["title_key"],
    base["release_year"].fillna(-1).astype(int)
))
mask_not_old_title_year = ~pd.Series(new_title_year, index=base.index).isin(old_title_year)

append_df = base[
    mask_not_old_id & mask_not_old_imdb & mask_not_old_title_year
].copy()

append_df = append_df.drop(columns=["title_key"], errors="ignore")
old_master = old_master.drop(columns=["title_key"], errors="ignore")

print("Rows to append:", append_df.shape)

# ------------------------------------------------------------
# 8. CONCAT TO MAKE V3
# ------------------------------------------------------------
print("\n8. Building thesis_master_v3...")

master_v3 = pd.concat([old_master, append_df], ignore_index=True)

master_v3["id"] = to_num(master_v3["id"])
master_v3 = master_v3.dropna(subset=["id"]).copy()
master_v3["id"] = master_v3["id"].astype("int64")
master_v3 = master_v3.drop_duplicates(subset="id").sort_values("id").reset_index(drop=True)

print("Final thesis_master_v3 shape:", master_v3.shape)

# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------
print("\n9. Saving...")
master_v3.to_csv(OUTPUT_PATH, index=False)

print("\n✅ DONE")
print("Saved:", OUTPUT_PATH)
print("Old master rows:", len(old_master))
print("Strict appended rows:", len(append_df))
print("Final rows:", len(master_v3))

# ------------------------------------------------------------
# 10. DIAGNOSTICS
# ------------------------------------------------------------
print("\nYear coverage (tail):")
print(master_v3["release_year"].value_counts(dropna=True).sort_index().tail(20))

print("\nMissing values (%) in thesis_master_v3:")
print((master_v3.isna().mean() * 100).round(2).sort_values(ascending=False))

print("\nSample of appended rows:")
display(append_df.head(10))

Mounted at /content/drive
1. Loading old master...
Old master shape: (45430, 22)

2. Loading new dataset...
New dataset shape: (1072255, 42)

3. Applying base release/year filters...
After release/year filter: (288828, 42)

4. Standardizing fields...

5. Applying strict quality filters...
After strict filters: (44660, 48)

6. Building text columns...

7. Removing overlaps with old master...
Rows to append: (44139, 22)

8. Building thesis_master_v3...
Final thesis_master_v3 shape: (89569, 22)

9. Saving...

✅ DONE
Saved: /content/drive/MyDrive/thesis_movie_project/processed/thesis_master_final.csv
Old master rows: 45430
Strict appended rows: 44139
Final rows: 89569

Year coverage (tail):
release_year
2006.0    1269
2007.0    1319
2008.0    1470
2009.0    1585
2010.0    1501
2011.0    1666
2012.0    1721
2013.0    1887
2014.0    1973
2015.0    1904
2016.0    1604
2017.0    5941
2018.0    6155
2019.0    6483
2020.0    5702
2021.0    6202
2022.0    7348
2023.0    6081
2024.0     764
2025.0

,id,movieId,imdb_id,title,overview,keywords_clean,cast_clean,director_name,genres_clean,original_language,...,budget,revenue,roi_raw,runtime,popularity,vote_average,vote_count,text_overview_only,text_semantic_enriched,text_semantic_full
6,299536,<NA>,tt4154756,Avengers: Infinity War,As the Avengers and their allies have continue...,sacrifice magic superhero based on comic space...,Gerard Villegas Sebastián Haro Rosana Pastor,"Anthony Russo, Joe Russo",Adventure Action Science Fiction,en,...,300000000.0,2.052415e+09,6.841383,149.0,154.340,8.255,27713,As the Avengers and their allies have continue...,Avengers: Infinity War sacrifice magic superhe...,Avengers: Infinity War sacrifice magic superhe...
15,299534,<NA>,tt4154796,Avengers: Endgame,After the devastating events of Avengers: Infi...,superhero time travel space travel time machin...,John C. Reilly Alexis Denisof Lloyd Kaufman Dj...,"Anthony Russo, Joe Russo",Adventure Science Fiction Action,en,...,356000000.0,2.800000e+09,7.865169,181.0,91.756,8.263,23857,After the devastating events of Avengers: Infi...,Avengers: Endgame superhero time travel space ...,Avengers: Endgame superhero time travel space ...
18,475557,<NA>,tt7286456,Joker,"During the 1980s, a failed stand-up comedian i...",dream street gang society psychopath clown vil...,Will Theakston Maggie Smith Violet Columbus Ol...,Todd Phillips,Crime Thriller Drama,en,...,55000000.0,1.074458e+09,19.535605,122.0,54.522,8.168,23425,"During the 1980s, a failed stand-up comedian i...",Joker dream street gang society psychopath clo...,Joker dream street gang society psychopath clo...
27,284054,<NA>,tt1825683,Black Panther,"King T'Challa returns home to the reclusive, t...",africa superhero based on comic aftercreditsst...,Unknown,Ryan Coogler,Action Adventure Science Fiction,en,...,200000000.0,1.349926e+09,6.749630,135.0,43.665,7.390,21053,"King T'Challa returns home to the reclusive, t...",Black Panther africa superhero based on comic ...,Black Panther africa superhero based on comic ...
32,315635,<NA>,tt2250912,Spider-Man: Homecoming,Following the events of Captain America: Civil...,high school new york city washington dc usa su...,Johnny Martin Jonathan Evans-Jones Tricia O'Ne...,Jon Watts,Action Adventure Science Fiction Drama,en,...,175000000.0,8.801669e+08,5.029525,133.0,65.880,7.345,20507,Following the events of Captain America: Civil...,Spider-Man: Homecoming high school new york ci...,Spider-Man: Homecoming high school new york ci...
57,634649,<NA>,tt10872600,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,new york city hero showdown magic loss of love...,Beulah Hashe Chan Woo Lim Robert Downey Jr. Je...,Jon Watts,Action Adventure Science Fiction,en,...,200000000.0,1.921847e+09,9.609236,148.0,186.065,7.990,18299,Peter Parker is unmasked and no longer able to...,Spider-Man: No Way Home new york city hero sho...,Spider-Man: No Way Home new york city hero sho...
62,346364,<NA>,tt1396484,It,"In a small town in Maine, seven children known...",small town based on novel or book clown bullyi...,Alex R. Hibbert Alex C. Riley Hughes Shamel He...,Andy Muschietti,Horror Fantasy,en,...,40000000.0,7.018000e+08,17.545000,135.0,74.730,7.242,18018,"In a small town in Maine, seven children known...",It small town based on novel or book clown bul...,It small town based on novel or book clown bul...
66,354912,<NA>,tt2380307,Coco,Despite his family’s baffling generations-old ...,skeleton mexico guitar afterlife musician holi...,Pezh Maan Alaa Safi Juani Feliz Topo Wresniwir...,"Lee Unkrich, Adrian Molina",Family Animation Fantasy Music Comedy Adventure,en,...,175000000.0,8.005260e+08,4.574434,105.0,166.578,8.222,17742,Despite his family’s baffling generations-old ...,Coco skeleton mexico guitar afterlife musician...,Coco skeleton mexico guitar afterlife musician...
81,496243,<NA>,tt6751668,Parasite,"All unemployed, Ki-taek's family takes peculia...",dark comedy private lessons birthday party con...,Christopher Ma

In [2]:
import pandas as pd
import numpy as np

FINAL_PATH = "/content/drive/MyDrive/thesis_movie_project/processed/thesis_master_final.csv"
master = pd.read_csv(FINAL_PATH, low_memory=False)

placeholder_values = {
    "Unknown", "unknown", "UNKNOWN",
    "nan", "NaN", "None", "NONE",
    "N/A", "n/a", "null", "NULL"
}

text_cols = [
    "overview",
    "keywords_clean",
    "cast_clean",
    "director_name",
    "genres_clean",
    "text_overview_only",
    "text_semantic_enriched",
    "text_semantic_full"
]

for col in text_cols:
    if col in master.columns:
        master[col] = master[col].fillna("").astype(str).str.strip()
        master[col] = master[col].replace(list(placeholder_values), "")

# Rebuild text columns after placeholder cleaning
master["text_overview_only"] = master["overview"].fillna("").astype(str).str.strip()

master["text_semantic_enriched"] = (
    master["title"].fillna("").astype(str).str.strip() + " " +
    master["keywords_clean"].fillna("").astype(str).str.strip() + " " +
    master["overview"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

master["text_semantic_full"] = (
    master["title"].fillna("").astype(str).str.strip() + " " +
    master["keywords_clean"].fillna("").astype(str).str.strip() + " " +
    master["cast_clean"].fillna("").astype(str).str.strip() + " " +
    master["overview"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

master.to_csv(FINAL_PATH, index=False)

print("Saved cleaned thesis_master_final.csv")
print((master[text_cols].replace('', np.nan).isna().mean() * 100).round(2))

Saved cleaned thesis_master_final.csv
overview                   1.07
keywords_clean            16.01
cast_clean                14.12
director_name              4.45
genres_clean               6.66
text_overview_only         1.07
text_semantic_enriched     0.00
text_semantic_full         0.00
dtype: float64


In [3]:
df_semantic_core = master[
    (master["text_overview_only"].fillna("").astype(str).str.len() > 30) &
    (pd.to_numeric(master["runtime"], errors="coerce").fillna(0) >= 60) &
    (pd.to_numeric(master["release_year"], errors="coerce").notna())
].copy()

In [4]:
df_network_compare = df_semantic_core.copy()

In [5]:
master["revenue_clean"] = pd.to_numeric(master["revenue"], errors="coerce")
master.loc[master["revenue_clean"] <= 0, "revenue_clean"] = np.nan

df_revenue = df_semantic_core[
    master.loc[df_semantic_core.index, "revenue_clean"].notna()
].copy()

In [6]:
master["budget_clean"] = pd.to_numeric(master["budget"], errors="coerce")
master.loc[master["budget_clean"] <= 0, "budget_clean"] = np.nan

df_roi = df_semantic_core[
    master.loc[df_semantic_core.index, "budget_clean"].notna() &
    master.loc[df_semantic_core.index, "revenue_clean"].notna()
].copy()

In [7]:
import pandas as pd
import numpy as np

PATH = "/content/drive/MyDrive/thesis_movie_project/processed/thesis_master_final.csv"
master = pd.read_csv(PATH, low_memory=False)

# Basic helpers
master["overview_len"] = master["overview"].fillna("").astype(str).str.len()
master["overview_word_count"] = master["overview"].fillna("").astype(str).str.split().str.len()

master["budget_clean"] = pd.to_numeric(master["budget"], errors="coerce")
master["revenue_clean"] = pd.to_numeric(master["revenue"], errors="coerce")

master.loc[master["budget_clean"] <= 0, "budget_clean"] = np.nan
master.loc[master["revenue_clean"] <= 0, "revenue_clean"] = np.nan

master["roi_clean"] = np.where(
    master["budget_clean"].notna() & master["revenue_clean"].notna(),
    master["revenue_clean"] / master["budget_clean"],
    np.nan
)

# 1. Main semantic core
df_semantic_core = master[
    (master["text_overview_only"].fillna("").astype(str).str.len() > 30) &
    (pd.to_numeric(master["runtime"], errors="coerce").fillna(0) >= 60) &
    (pd.to_numeric(master["release_year"], errors="coerce").notna())
].copy()

# 2. Representation comparison dataset
df_network_compare = df_semantic_core.copy()

# 3. Revenue validation dataset
df_revenue = df_semantic_core[
    master.loc[df_semantic_core.index, "revenue_clean"].notna()
].copy()
df_revenue["log_revenue"] = np.log1p(df_revenue["revenue_clean"])

# 4. ROI validation dataset
df_roi = df_semantic_core[
    master.loc[df_semantic_core.index, "budget_clean"].notna() &
    master.loc[df_semantic_core.index, "revenue_clean"].notna()
].copy()
df_roi["log_budget"] = np.log1p(df_roi["budget_clean"])
df_roi["log_revenue"] = np.log1p(df_roi["revenue_clean"])

print("df_semantic_core:", df_semantic_core.shape)
print("df_network_compare:", df_network_compare.shape)
print("df_revenue:", df_revenue.shape)
print("df_roi:", df_roi.shape)

SAVE_DIR = "/content/drive/MyDrive/thesis_movie_project/processed"
df_semantic_core.to_csv(f"{SAVE_DIR}/df_semantic_core.csv", index=False)
df_network_compare.to_csv(f"{SAVE_DIR}/df_network_compare.csv", index=False)
df_revenue.to_csv(f"{SAVE_DIR}/df_revenue.csv", index=False)
df_roi.to_csv(f"{SAVE_DIR}/df_roi.csv", index=False)

print("Saved all final subsets.")

df_semantic_core: (84882, 27)
df_network_compare: (84882, 27)
df_revenue: (10439, 28)
df_roi: (6927, 29)
Saved all final subsets.
